In [1]:
import polars as pl
import math

In [2]:
DATA_PATH = "S:/campaignfinance/data/"
cand_contrib = pl.read_parquet(f"{DATA_PATH}FEC/processed/firm_candidate_contributions.parquet")
cmte_contrib = pl.read_parquet(f"{DATA_PATH}FEC/processed/firm_committee_contributions.parquet")
principal_cmtes_H = pl.read_csv(f"{DATA_PATH}FEC/processed/candidate_principal_committees_H.csv")
principal_cmtes_S = pl.read_csv(f"{DATA_PATH}FEC/processed/candidate_principal_committees_S.csv")
leadership_cmtes = pl.read_csv(f"{DATA_PATH}FEC/processed/leadership_committees.csv")

In [3]:
keep_types = ["24K", "24Z", "24N", "24U", "24P", "24R"]

In [4]:
principal_cmtes_H = (
    principal_cmtes_H
    .filter(pl.col("candidate_ids").str.starts_with("H"))
    .select(["committee_id", "cycle", "candidate_id"])
    .rename({"committee_id":"rcpt_cmte", "candidate_id":"matched_cand"})
    .unique()
)

principal_cmtes_S = (
    principal_cmtes_S
    .filter(pl.col("candidate_ids").str.starts_with("S"))
    .select(["committee_id", "cycle", "candidate_id"])
    .rename({"committee_id":"rcpt_cmte", "candidate_id":"matched_cand"})
    .unique()
)

leadership_cmtes = (
    leadership_cmtes
    .filter(pl.col("sponsor_candidate_ids").str.starts_with("H"))
    .select(["committee_id", "cycle", "sponsor_candidate_ids"])
    .rename({"committee_id":"rcpt_cmte", "sponsor_candidate_ids":"matched_cand"})
    .unique()
)

cand_cmtes = pl.concat([principal_cmtes_H, principal_cmtes_S])

cand_cmtes = cand_cmtes.with_columns(
    pl.lit("P").alias("cmte_type")
)
leadership_cmtes = leadership_cmtes.with_columns(
    pl.lit("D").alias("cmte_type")
)

In [5]:
cand_contrib = cand_contrib.filter(pl.col("transaction_type").is_in(keep_types))

cand_contrib = cand_contrib.with_columns(([
    pl.col("rcpt_cmte").fill_null(pl.col("cand_id")).alias("rcpt_cmte"),
    pl.col("cand_id").fill_null(pl.col("rcpt_cmte")).alias("cand_id")
]))

cand_contrib = cand_contrib.with_columns([
    pl.when(pl.col("rcpt_cmte").str.contains(r"^C"))
    .then(pl.col("rcpt_cmte"))
    .otherwise(None)
    .alias("rcpt_cmte"),
    pl.when(pl.col("cand_id").str.contains(r"^[PHS]"))
    .then(pl.col("cand_id"))
    .otherwise(None)
    .alias("cand_id"),
    (((pl.col("year").cast(pl.Int64) + 1) // 2) * 2 ).alias("cycle")
])

cmte_contrib = cmte_contrib.filter(pl.col("transaction_type").is_in(keep_types))

cmte_contrib = cmte_contrib.with_columns(
    pl.col("rcpt_cmte").alias("cand_id")
)

cmte_contrib = cmte_contrib.with_columns([
    pl.when(pl.col("rcpt_cmte").str.contains(r"^C"))
    .then(pl.col("rcpt_cmte"))
    .otherwise(None)
    .alias("rcpt_cmte"),
    pl.when(pl.col("cand_id").str.contains(r"^[PHS]"))
    .then(pl.col("cand_id"))
    .otherwise(None)
    .alias("cand_id"),
    (((pl.col("year").cast(pl.Int64) + 1) // 2) * 2 ).alias("cycle")
])

In [ ]:
cand_cand_contrib = cand_contrib.join(cand_cmtes, on = ["rcpt_cmte", "cycle"], how = "left")
cand_cand_contrib = (
    cand_cand_contrib
    .with_columns([
        pl.col("matched_cand").fill_null(pl.col("cand_id")).alias("matched_cand")
        ])
    .drop_nulls(subset = ["matched_cand", "cmte_type"])
    .drop("cand_id")
    .rename({"matched_cand": "cand_id"})
)

cand_cmte_contrib = cmte_contrib.join(cand_cmtes, on = ["rcpt_cmte", "cycle"], how = "left")
cand_cmte_contrib = (
    cand_cmte_contrib
    .with_columns([
        pl.col("matched_cand").fill_null(pl.col("cand_id")).alias("matched_cand")
        ])
    .drop_nulls(subset = ["matched_cand", "cmte_type"])
    .drop("cand_id")
    .rename({"matched_cand": "cand_id"})
)

full_cand_contrib = pl.concat([cand_cand_contrib, cand_cmte_contrib]).unique(maintain_order=True)
full_cand_contrib = full_cand_contrib.filter(pl.col("amount") > 0)

In [45]:
lead_cand_contrib = cand_contrib.join(leadership_cmtes, on = ["rcpt_cmte", "cycle"], how = "left")
lead_cand_contrib = (
    lead_cand_contrib
    .with_columns([
        pl.col("matched_cand").fill_null(pl.col("cand_id")).alias("matched_cand")
    ])
    .drop_nulls(subset = ["matched_cand", "cmte_type"])
    .drop("cand_id")
    .rename({"matched_cand": "cand_id"})
)

lead_cmte_contrib = cmte_contrib.join(leadership_cmtes, on = ["rcpt_cmte", "cycle"], how = "left")
lead_cmte_contrib = (
    lead_cmte_contrib
    .with_columns([
        pl.col("matched_cand").fill_null(pl.col("cand_id")).alias("matched_cand")
    ])
    .drop_nulls(subset = ["matched_cand", "cmte_type"])
    .drop("cand_id")
    .rename({"matched_cand": "cand_id"})
)

full_leadership_contrib = pl.concat([lead_cand_contrib, lead_cmte_contrib]).unique(maintain_order=True)
full_leadership_contrib = full_leadership_contrib.filter(pl.col("amount") > 0)

In [46]:
full_leadership_contrib

cmte_id,transaction_type,name,city,state,zip,amount,rcpt_cmte,memo,year,month,day,cmte_nm,connected_org,corporation,parent,website,instrument,cusip,isin,common_name,business_description,ric,former_name,business_name,hq,hq_state,hq_city,permid,parent_hq,parent_permid,GICS_ID,GICS_Sector,GICS_Industry_Group,GICS_Industry,GIS_Subindustry,ICB_ID,ICB_Industry,ICB_Super_Sector,ICB_Sector,ICB_Sub_Sector,Incorporation_Date,Public_Date,IPO_Date,Retire_Date,Inactive_Year,Inactive_Event,Listing_Status,TRBC_ID,TRBC_Econ_Sector,TRBC_Business_Sector,TRBC_Industry_Group,TRBC_Industry,TRBC_Activity,NAICS_ID,NAICS_Sector,NAICS_Subsector,NAICS_Industry_Group,NAICS_International_Industry,NAICS_National_Industry,cycle,cand_id,cmte_type
str,str,str,str,str,str,f64,str,str,i64,i64,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,f64,str,f64,f64,str,str,str,str,f64,str,str,str,str,str,str,str,str,f64,f64,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str
"""C00236414""","""24K""","""AMERIPAC""","""WASHINGTON""","""DC""","""20003""",2000.0,"""C00271338""","""M-MD-5TH DIST.(BREAKFAST 7/20/…",2004,5,20,"""AEGON USA INC POLITICAL ACTION…","""AEGON USA INC""","""AEGON USA""","""AEGN.AS""",null,"""4295911673""",null,null,"""Transamerica Corp""","""Transamerica Corp (Transameric…",null,null,null,"""United States of America""","""IOWA""","""Cedar Rapids""",4.2959e9,"""Netherlands""",5.0868e9,null,null,null,null,null,null,null,null,null,null,"""1999-02-16""",null,null,null,null,null,null,"""['5530103011', '5530103010', '…","""['Financials', 'Financials', '…","""['Insurance', 'Insurance', 'In…","""['Insurance', 'Insurance', 'In…","""['Life & Health Insurance', 'L…","""['Life Insurance', 'Life & Hea…","""['524113', '551112']""","""['Finance and Insurance', 'Man…","""['Insurance Carriers and Relat…","""['Insurance Carriers', 'Manage…","""['Direct Life, Health, and Med…","""['Direct Life Insurance Carrie…",2004,"""H2MD05155""","""D"""
"""C00236414""","""24K""","""AMERICAN SUCCESS PAC""","""WASHINGTON""","""DC""","""20036""",1000.0,"""C00336644""","""R-CA-26TH DIST.(LUNCH 6/15/04)""",2004,5,20,"""AEGON USA INC POLITICAL ACTION…","""AEGON USA INC""","""AEGON USA""","""AEGN.AS""",null,"""4295911673""",null,null,"""Transamerica Corp""","""Transamerica Corp (Transameric…",null,null,null,"""United States of America""","""IOWA""","""Cedar Rapids""",4.2959e9,"""Netherlands""",5.0868e9,null,null,null,null,null,null,null,null,null,null,"""1999-02-16""",null,null,null,null,null,null,"""['5530103011', '5530103010', '…","""['Financials', 'Financials', '…","""['Insurance', 'Insurance', 'In…","""['Insurance', 'Insurance', 'In…","""['Life & Health Insurance', 'L…","""['Life Insurance', 'Life & Hea…","""['524113', '551112']""","""['Finance and Insurance', 'Man…","""['Insurance Carriers and Relat…","""['Insurance Carriers', 'Manage…","""['Direct Life, Health, and Med…","""['Direct Life Insurance Carrie…",2004,"""H8CA35032""","""D"""
"""C00236414""","""24K""","""TOMPAC""","""ARLINGTON""","""VA""","""22215""",2500.0,"""C00364174""","""R-NY-26TH DIST. (DINNER 5/12/0…",2004,5,13,"""AEGON USA INC POLITICAL ACTION…","""AEGON USA INC""","""AEGON USA""","""AEGN.AS""",null,"""4295911673""",null,null,"""Transamerica Corp""","""Transamerica Corp (Transameric…",null,null,null,"""United States of America""","""IOWA""","""Cedar Rapids""",4.2959e9,"""Netherlands""",5.0868e9,null,null,null,null,null,null,null,null,null,null,"""1999-02-16""",null,null,null,null,null,null,"""['5530103011', '5530103010', '…","""['Financials', 'Financials', '…","""['Insurance', 'Insurance', 'In…","""['Insurance', 'Insurance', 'In…","""['Life & Health Insurance', 'L…","""['Life Insurance', 'Life & Hea…","""['524113', '551112']""","""['Finance and Insurance', 'Man…","""['Insurance Carriers and Relat…","""['Insurance Carriers', 'Manage…","""['Direct Life, Health, and Med…","""['Direct Life Insurance Carrie…",2004,"""H4WI04134""","""D"""
"""C00137828""","""24K""","""CMTE FOR THE PRESERVAT

In [47]:
cand_contrib.shape

(519801, 62)

In [2]:
DATA_PATH = "S:/campaignfinance/data/"
full_cand_contrib = pl.read_parquet(f"{DATA_PATH}FEC/processed/firms_principal_committees_contributions.parquet")

In [7]:
print(full_cand_contrib.shape)
print(full_cand_contrib.filter((pl.col("year") % 2 == 0) & (pl.col("month") < 11)).shape)
print(full_cand_contrib.filter((pl.col("year") % 2 == 0) & (pl.col("month") > 6) & (pl.col("month") < 11)).shape)


(496238, 63)
(478424, 63)
(223391, 63)


### Transaction Types to Keep and Explanations
- ``24K``: Contribution made to nonaffiliated committee
- ``24Z``: In-kind contribution made to registered filer (in-kind contributions such as service/goods provided)
- ``24N``: Communication cost against candidate (not super common but should be treated as money given to candidates), ``rcpt_cmte`` column records actual candidate_id and not cmte_id
- ``24U``: Contribution made to unregistered entity (there may be some coding error here. There are only 4 entries, so only keep ones with valid ``rcpt_cmte``)
- ``24R``: Recount disbursement
- ``24E``: Independent expenditure advocating election of candidate
- ``24A``: Independent expenditure opposing election of candidate
- ``24P``: Contribution made to possible federal candidate including in-kind contributions (need to match name to candidate commttee names)